## 1. Import Libraries

- **pandas** : load and handle the dataset
- **train_test_split** : split data into training and testing sets
- **CountVectorizer**:convert text messages into word counts
- **MultinomialNB**: Naive Bayes model for text classification
- **accuracy_score / classification_report** :evaluate the model

In [84]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## 2. Load the Dataset

- **`encoding='latin-1'`**: I needed that because the file contains special characters that default encoding (UTF-8) can't read properly.

The dataset is now stored in `df`.

In [85]:
df = pd.read_csv(r"C:\Users\Junayed\spam-detection-model\spam_data.csv", encoding='latin-1')

In [86]:
df.head(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 3. Keep Only Needed Columns

- **`v1`**: the label (`ham` or `spam`)
- **`v2`**: the message text

`.head(5)` shows the first 5 rows to confirm it looks right.

In [87]:
df = df[['v1', 'v2']]
df.head(5)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 4. Rename Columns

The original column names `v1` and `v2` aren't clear, so need to  rename them:

- **`v1` → `label`**: spam or ham
- **`v2` → `message`**: the SMS text

Now the columns are easy to understand.

In [88]:
df.columns = ['label', 'message']

In [89]:
df.head(5)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 5. Check Class Distribution

- **ham:** 4825 (≈87%)
- **spam:** 747 (≈13%)

The data is **imbalanced**; most messages are ham. This means accuracy alone can be misleading (a model that always predicts "ham" would still score ~87%). We'll need to look at **precision and recall** for spam later.

In [90]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

## 5.1: Remove the duplicates


Duplicate messages can bias the model, it may memorize repeated text instead of learning real patterns.

- **Before:** 5572 messages
- **After:** 5169 messages
- **Removed:** 403 duplicates

`drop_duplicates(subset="message")` to keep only unique messages, then reset the index so rows stay clean and sequential.

In [91]:
print("Before:", len(df))
df = df.drop_duplicates(subset="message").reset_index(drop=True)
print("After:", len(df))

Before: 5572
After: 5169


## 6. Convert Labels to Numbers

ML models can't work with text labels, so need to  convert them to numbers:

- **spam → 1**
- **ham → 0**

We do this with a small function applied to every row, and store the result in a new column **`label_dif`**.

The output confirms it worked, spam rows now show `1`, ham rows show `0`.

In [92]:
def label_diff(label):
    if label == "spam":
        return 1
    else:
        return 0

df["label_dif"] = df["label"].apply(label_diff)
df.head(5)

,label,message,label_dif
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 7. Clean the Text

Raw text contains noise that hurts the model: uppercase letters, numbers, and punctuation. We write a `clean_text()` function that:

1. **Lowercases** everything :`Free` → `free`
2. **Removes numbers**: `2`, `500`, `1000`
3. **Removes punctuation**: `!`, `?`, `.`, `,`
4. **Strips extra whitespace**

Then apply it to every message and store the result in a new column **`clean_message`**.

### Why this matters

Without cleaning, the model treats `FREE`, `Free`, and `free!` as **three different words**. Cleaning merges them into one → the model generalizes better on **new, unseen messages**.

Looking at the output: `"Free entry in 2 a wkly comp to win FA Cup..."` becomes `"free entry in a wkly comp to win fa cup..."` — simpler, consistent, and ready for vectorizing.

In [93]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

df["clean_message"] = df["message"].apply(clean_text)
df[["message", "clean_message"]].head(5)

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


## 8. Convert Text to Numbers

now convert the cleaned messages into numbers using **`TfidfVectorizer`** instead of `CountVectorizer`.

### What changed and why

- **`CountVectorizer`** → just counts words. `"free"` appearing once = `1`. A rare spam word and a common word get the same weight if they appear the same number of times.
- **`TfidfVectorizer`** → weighs words by **importance**. Rare-but-meaningful words (like `"claim"`, `"urgent"`, `"prize"`) get **higher scores**; words that appear in almost every message get **lower scores**.

This helps the model **detect spam in new, unseen messages** not just messages it has memorized.

### Settings

- **`stop_words='english'`**: removes common words like "the", "is"
- **`max_features=5000`** : keeps only the 5000 most informative words
- **Input**: `df['clean_message']` (cleaned text, not raw)

**Result:**
- **`X`** :messages as TF-IDF vectors
- **`Y`** :target labels (0 = ham, 1 = spam)

In [94]:
# cv = CountVectorizer(stop_words='english', max_features=3000)

# X = cv.fit_transform(df['message'])
# Y = df['label_dif']



from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english', max_features=5000)
X = cv.fit_transform(df['clean_message'])
Y = df['label_dif']

## 9. Split Data into Train and Test

- **Training set (80%)**: the model learns from this
- **Test set (20%)**: we use this to check how well the model performs on unseen data

- **`random_state=43`**: makes the split reproducible (same split every run)

In [95]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state= 43)

## 10. Train the Model & Evaluate

We train **Multinomial Naive Bayes** on the TF-IDF features, then predict on the test set.

### Results

- **Accuracy: 96.52%**: slightly lower than the old 98.39%, but **more honest**.

| Class | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| Ham (not spam) | 0.96 | 1.00 | 0.98 | 895 |
| Spam | 1.00 | 0.74 | 0.85 | 139 |

### What this means

- **Precision (spam) = 1.00** → Every message the model flagged as spam **was actually spam**. Zero false alarms.
- **Recall (spam) = 0.74** → It caught 74% of real spam. It **missed about 26%** of spam messages.
- **F1 (spam) = 0.85** → Solid balance, but recall is the weak side.

### Why accuracy dropped (and why this is better)

The old **98.39%** looked nicer, but it was partly **memorization** raw counts let the model rely on exact words and couldn't generalize to new messages.

After **cleaning + TF-IDF**, accuracy dropped a bit, but the model now:

- Generalizes to **unseen messages**
- Has **zero false positives** (never marks a real message as spam)
- Misses ~1 in 4 spam messages


In [96]:
model = MultinomialNB()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(Y_test, Y_pred):.2%}")
print(classification_report(Y_test, Y_pred,
target_names=["Ham (not spam)", 'spam']))

Accuracy: 96.52%
                precision    recall  f1-score   support

Ham (not spam)       0.96      1.00      0.98       895
          spam       1.00      0.74      0.85       139

      accuracy                           0.97      1034
     macro avg       0.98      0.87      0.92      1034
  weighted avg       0.97      0.97      0.96      1034



## 10. Test the Model on New Messages

built a small helper function `check_spam()` that:

1. **Takes a new message** as input
2. **Converts it** into word counts using the same `CountVectorizer` (`cv.transform`)
3. **Predicts** the label (0 = ham, 1 = spam)
4. **Prints the probability** so we see how confident the model is

### Example

**Input:** `"Congratulations! You won a free iPhone. Click now!"`  
**Output:** `Spam (Probability: 99.77%)`

In [104]:
def check_spam(message):
    cleaned = clean_text(message)
    msg_vec = cv.transform([cleaned])
    pred = model.predict(msg_vec)[0]
    prob = model.predict_proba(msg_vec)[0]
    label = 'Spam' if pred == 1 else "Not spam"
    print(f"Message: {message}")
    print(f"Prediction: {label} (Probability: {prob[pred]:.2%})")

check_spam("Sunshine Quiz Wkly Q! Win a top Sony DVD player if u know which country the Algarve is in? Txt ansr to 82277. å£1.50 SP:Tyrone")

Message: Sunshine Quiz Wkly Q! Win a top Sony DVD player if u know which country the Algarve is in? Txt ansr to 82277. å£1.50 SP:Tyrone
Prediction: Spam (Probability: 92.08%)


In [98]:
check_spam('Hey, are we still meeting tomorrow at 3pm?')
check_spam('URGENT: Your account has been compromised. Call us!')


Message: Hey, are we still meeting tomorrow at 3pm?
Prediction: Not spam (Probability: 99.28%)
Message: URGENT: Your account has been compromised. Call us!
Prediction: Spam (Probability: 61.75%)


In [99]:
check_spam("As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune")

check_spam("Todays Voda numbers ending 7548 are selected to receive a $350 award. If you have a match please call 08712300220 quoting claim code 4041 standard rates app")

Message: As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
Prediction: Not spam (Probability: 87.46%)
Message: Todays Voda numbers ending 7548 are selected to receive a $350 award. If you have a match please call 08712300220 quoting claim code 4041 standard rates app
Prediction: Spam (Probability: 98.61%)


In [100]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model,f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(cv,f)

print("saved.")

saved.
